# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Lane 4: CTR / Engagement Opportunity Scoring.**

**Abstract (question -> data -> method -> headline result -> what it's for):** Among search
pages that already rank well enough to earn clicks, which ones get meaningfully fewer clicks
than other pages at the same position, and how should a content team prioritize reviewing them?
Using FlyRank's anonymized 30,000-row ML Internship starter dataset, a transparent baseline
ranks pages by how far their CTR falls below the median of position-tier peers, and a Random
Forest, deliberately excluding CTR itself, tests whether content and engagement signals alone
can anticipate underperformance. The position-tier pattern is strong (CTR falls from 0.35% at
page one to 0.055% deep, thousands of pages per tier); the honest model reaches ROC AUC 0.727
against a 0.500 floor, a real but modest signal led by on-page engagement rate. This is a
decision-support ranking tool for a content editor's weekly review queue, not a causal or
predictive claim about what a rewrite would achieve.

**Decision this supports:** which pages an SEO content editor reviews first for a title, meta
description, or snippet rewrite, out of a much larger pool, given limited review time.

**Unit of analysis:** one page (`content_id`), one 90-day observation window.

**Action:** an editor works down a ranked review queue and rewrites the title/meta/snippet for
the pages at the top first.

**Cost of a wrong call:** review time spent on a page whose low CTR is actually volume noise or
a data-quality artifact, not a real content problem, this project treats a minimum-volume floor
as part of the method for exactly that reason.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/jasleen13/ML-Internship.git"
DATA_REL = Path("data") / "raw" / "content_refresh_anonymized.csv"

def find_repo_root(start: Path):
    p = start
    while not (p / DATA_REL).exists() and p != p.parent:
        p = p.parent
    return p if (p / DATA_REL).exists() else None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    clone_dir = Path("/content/ML-Internship") if Path("/content").exists() else Path.cwd() / "ML-Internship"
    if not (clone_dir / DATA_REL).exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

DATA_PATH = repo_root / DATA_REL
FIG_DIR = repo_root / "work" / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Using data at:", DATA_PATH)
print(df.shape)

visible = df["impressions_90d"] >= 500
in_range = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
lane = df[visible & in_range].copy()
print(f"Candidate pool: {len(lane):,} of {len(df):,} total pages")


Using data at: /home/claude/ML-Internship/data/raw/content_refresh_anonymized.csv
(30000, 44)
Candidate pool: 12,023 of 30,000 total pages


## 2. Data

**Primary release used:** the anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`,
30,000 rows), FlyRank's ML Internship program. Every column is an observable search or content
signal (impressions, clicks, position, CTR, content metadata), no product decision flags
(`health_score`, `priority_score`, `action_type`) are shipped in this release.

**Date window:** a rolling 90-day performance window per row (`impressions_90d`, `clicks_90d`,
`ctr`, etc.), the same window standard used throughout the FlyRank ML Internship program.

**A second release was also explored:** the full warehouse (`FlyRank/internship-warehouse` on
Hugging Face, gated), queried via DuckDB against a mid-panel month (`month=2026-03`) to build a
monthly-grain data contract and a deliberate leakage demonstration. That work lives in
`w03_data_contract.ipynb`; the starter CSV is the primary dataset for every modeling week
because it's the release this capstone's model and baseline are built and validated on.

**Excluded, with why:** `fact_content_query_90d` from the warehouse (a fixed trailing 90-day
window that overlaps multiple calendar months, not aligned to the single-month grain used in the
data contract). Within the starter CSV, `trend_direction` and `trend_pct` are excluded from
every feature set, they describe a 30-day-vs-previous-30-day change and are reserved as label
material only, never as model inputs, per the leakage lessons practiced in Weeks 2, 3, and 6.

**Public-safe:** no client names, domains, URLs, or raw queries appear anywhere in this dataset
or this analysis, IDs are pre-scrambled by FlyRank before release.


In [2]:
print("Columns used as features (never as the label source):")
print(sorted(["impressions_90d", "avg_position", "word_count", "content_age_days",
              "days_since_last_update", "engagement_rate", "scroll_rate", "ai_traffic_pct",
              "search_volume", "competition", "cpc", "content_type", "main_intent", "position_tier"]))
print()
print("Columns reserved as label material only, never used as features:")
print(["ctr", "ctr_gap (derived)", "clicks_90d", "trend_direction", "trend_pct"])


Columns used as features (never as the label source):
['ai_traffic_pct', 'avg_position', 'competition', 'content_age_days', 'content_type', 'cpc', 'days_since_last_update', 'engagement_rate', 'impressions_90d', 'main_intent', 'position_tier', 'scroll_rate', 'search_volume', 'word_count']

Columns reserved as label material only, never used as features:
['ctr', 'ctr_gap (derived)', 'clicks_90d', 'trend_direction', 'trend_pct']


## 3. Methodology

**Label / proxy:** `is_underperforming`, a page's observed CTR falls below the median CTR of
other pages in the same `position_tier`, computed from the same 90-day window as every feature.
This is a current-window descriptive proxy, not an observed future outcome, it says a page sits
below its peers right now, not that fixing it would change anything, that claim would need a
before/after experiment this dataset can't provide.

**Baseline:** a transparent rule (Week 4), score = `(ctr_gap < 0) x max(0, -ctr_gap) x
log1p(impressions_90d)`, reason code `low_ctr_visible_page`, action `rewrite_title_meta`.

**Features:** only signals knowable at the decision moment and never derived from the label:
`impressions_90d`, `avg_position`, `word_count`, `content_age_days`, `days_since_last_update`,
`engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `search_volume`, `competition`, `cpc`,
`content_type`, `main_intent`, `position_tier`. `ctr`, `ctr_gap`, and `clicks_90d` are excluded
from every model, since `is_underperforming` is a direct threshold on `ctr_gap`, including it
would be circular, not predictive (demonstrated explicitly in Weeks 5 and 6).

**Validation design:** client-grouped split (`GroupShuffleSplit` on `client_id`, 75/25), chosen
because pages from the same client share a template and strategy, a random split lets a model
partly memorize "this is client X's style" rather than learning something that generalizes.
Week 6 measured this directly: a plain random split leaked 27 of 28 clients across train/test
and inflated precision@20 to a suspicious 1.000; the grouped split (0 overlap) is the number
this capstone reports.

**Leakage checks:** the full attack checklist from `hunting-leakage-and-validating` was run
against the final feature set in Week 6, plus a direct confession test, re-adding `ctr_gap` as a
feature collapses ROC AUC from an honest 0.727 to a trivial 1.000, confirming it as the leak and
justifying its exclusion.


In [3]:
tier_median_ctr = lane.groupby("position_tier")["ctr"].transform("median")
lane["ctr_gap"] = lane["ctr"] - tier_median_ctr
lane["is_underperforming"] = (lane["ctr_gap"] < 0).astype(int)

print(f"Label base rate (is_underperforming): {lane['is_underperforming'].mean():.3f}")
print(f"Distinct clients in candidate pool: {lane['client_id'].nunique()}")


Label base rate (is_underperforming): 0.490
Distinct clients in candidate pool: 28


## 4. Results (vs baseline)

The baseline is not a fair target for the model to "beat", it's built directly from `ctr_gap`,
which is what the label is a threshold on, so it achieves near-perfect precision by
construction. The honest comparison is the model against a dummy majority-class floor, on the
same client-grouped split.


In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_honest = ["impressions_90d", "avg_position", "word_count", "content_age_days",
                   "days_since_last_update", "engagement_rate", "scroll_rate", "ai_traffic_pct",
                   "search_volume", "competition", "cpc"]
categorical_honest = ["content_type", "main_intent", "position_tier"]

model_df = lane.copy()
for c in numeric_honest:
    model_df[c] = model_df[c].fillna(model_df[c].median())
for c in categorical_honest:
    model_df[c] = model_df[c].fillna("unknown")

X = model_df[numeric_honest + categorical_honest]
y = model_df["is_underperforming"]
groups = model_df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_honest),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_honest),
])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return np.asarray(y_true)[order[:k]].mean()

rows = []
for name, clf in [
    ("dummy_majority", DummyClassifier(strategy="most_frequent", random_state=42)),
    ("logistic_regression", LogisticRegression(max_iter=2000)),
    ("decision_tree", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("random_forest", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
]:
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1] if hasattr(pipe, "predict_proba") else pipe.predict(X_te).astype(float)
    auc = roc_auc_score(y_te, proba) if len(set(y_te)) > 1 else float("nan")
    rows.append({"model": name, "roc_auc": round(auc, 3),
                 "precision_at_20": round(precision_at_k(y_te.values, proba, 20), 3),
                 "precision_at_50": round(precision_at_k(y_te.values, proba, 50), 3)})

results = pd.DataFrame(rows)
print(f"Base rate in test set: {y_te.mean():.3f}")
print(results.to_string(index=False))

results.to_csv(FIG_DIR.parent / "capstone_model_vs_baseline.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(results["model"], results["roc_auc"], color="#4C72B0")
ax.axhline(0.5, color="gray", linestyle="--", label="random guess floor")
ax.set_ylabel("ROC AUC (client-grouped split)")
ax.set_title("Model vs baseline floor: ROC AUC")
ax.set_xticklabels(results["model"], rotation=20, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "model_vs_baseline_auc.png", dpi=150)
plt.close()
print("Saved figure: model_vs_baseline_auc.png")


Base rate in test set: 0.441
              model  roc_auc  precision_at_20  precision_at_50
     dummy_majority    0.500             0.65             0.42
logistic_regression    0.545             0.75             0.64
      decision_tree    0.664             0.85             0.78
      random_forest    0.727             0.70             0.68
Saved figure: model_vs_baseline_auc.png


/tmp/ipykernel_628/4028394438.py:66: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(results["model"], rotation=20, ha="right")


## 5. Limitations

- **The label is a current-window proxy, not a future outcome.** `is_underperforming` describes
  where a page sits today relative to its peers, it doesn't say a rewrite would change anything,
  that requires a causal design (an actual before/after experiment) this dataset can't provide.
- **The baseline isn't a fair benchmark for the model.** It's built directly from `ctr_gap`,
  which the label thresholds on, so any "model beats baseline" framing would be misleading, the
  honest comparison here is against a dummy floor, not the baseline's circular precision.
- **The staleness signal is unresolved.** Week 4's check of `days_since_last_update` came back
  MIXED: the unfiltered direction is opposite the hypothesis (a volume artifact from n=174), the
  volume-floored direction matches it but at n=17, too thin to confirm either way, staleness was
  excluded from the model rather than guessed at.
- **Single-month warehouse work is a snapshot, not a trend.** The `w03` data contract used one
  mid-panel month, that can't separate a real CTR pattern from month-specific seasonality or
  mid-month client onboarding, this capstone's headline model uses the full-window starter CSV
  instead, for exactly that reason.
- **Observational, not causal.** Every claim in this project is directional and decision-support:
  it tells a reviewer where to look first, it does not prove that a specific edit will improve a
  specific page's performance.


In [5]:
floor = df[df["impressions_90d"] >= 500]
freshness_check = floor.groupby("freshness_tier").agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"))
print("Staleness check (visible pages only), the MIXED-verdict evidence from Week 4:")
print(freshness_check.sort_index())


Staleness check (visible pages only), the MIXED-verdict evidence from Week 4:
                    n  mean_ctr
freshness_tier                 
0-30            10063  0.271484
181+               17  0.208824
31-90              88  0.144886
91-180           6558  0.249686


## 6. Ranked recommendations

The full action playbook from Week 7: reason code `low_ctr_visible_page`, and a segment-level
action split by search intent (informational/navigational get `rewrite_title_meta`,
transactional/commercial get `improve_snippet_signals`, since price, rating, or availability
signals in the snippet plausibly move clicks more than title wording alone for those intents).

**No-go list, what should not be automated:** 1,215 pages in this queue show 0 clicks despite
real impression volume, exactly where a tracking or indexing artifact hides, not a content
problem, these need a manual check first. 736 pages sit at position 3 to 4, close enough to the
top 3 that a SERP feature (a snippet box, an "also asked" panel) could be suppressing clicks in
a way this dataset can't see, no title rewrite fixes that. No page in this queue should be
auto-rewritten without a human opening it first.

**Monitoring triggers:** candidate pool size or base rate drifting sharply from today's 12,023
pages / 0.490, the honest model's ROC AUC falling toward the 0.500 dummy floor on a fresh
client-grouped split, or the staleness signal finally reaching a large enough sample (n >= 50)
to move off MIXED, any of these should trigger a rebuild, not just a rerun.


In [6]:
def action_for(row):
    if row["main_intent"] in ("transactional", "commercial"):
        return "improve_snippet_signals"
    return "rewrite_title_meta"

lane["score"] = (lane["ctr_gap"] < 0).astype(int) * (-lane["ctr_gap"]).clip(lower=0) * np.log1p(lane["impressions_90d"])
lane["reason_code"] = "low_ctr_visible_page"
lane["action"] = lane.apply(action_for, axis=1)

queue = lane.sort_values("score", ascending=False).reset_index(drop=True)
print(f"Full ranked queue: {len(queue):,} candidate pages")
print()
print("Action mix:")
print(queue["action"].value_counts())
print()

zero_click = lane[lane["clicks_90d"] == 0]
near_top3 = lane[(lane["avg_position"] >= 3) & (lane["avg_position"] <= 4)]
print(f"No-go: zero-click pages = {len(zero_click):,}, position 3-4 SERP-risk pages = {len(near_top3):,}")

top10 = queue.head(10)[["content_id", "main_intent", "position_tier", "impressions_90d",
                          "avg_position", "ctr", "ctr_gap", "score", "reason_code", "action"]]
top10.to_csv(FIG_DIR.parent / "capstone_top10_recommendations.csv", index=False)
top10


Full ranked queue: 12,023 candidate pages

Action mix:
action
rewrite_title_meta         7252
improve_snippet_signals    4771
Name: count, dtype: int64

No-go: zero-click pages = 1,215, position 3-4 SERP-risk pages = 736


,content_id,main_intent,position_tier,impressions_90d,avg_position,ctr,ctr_gap,score,reason_code,action
0,content_c8e9d6ab9013,informational,page_1,208678,9.7,0.00,-0.24,2.939653,low_ctr_visible_page,rewrite_title_meta
1,content_453722754fea,informational,page_1,140079,7.6,0.01,-0.23,2.725493,low_ctr_visible_page,rewrite_title_meta
2,content_39881853ef0c,informational,page_1,112434,7.2,0.01,-0.23,2.674930,low_ctr_visible_page,rewrite_title_meta
3,content_c84a0ab98e90,informational,page_1,223271,7.8,0.03,-0.21,2.586391,low_ctr_visible_page,rewrite_title_meta
4,content_0919dd345d80,informational,page_1,119217,7.0,0.02,-0.22,2.571516,low_ctr_visible_page,rewrite_title_meta
5,content_d274ac4158ef,informational,page_1,65138,6.8,0.01,-0.23,2.549384,low_ctr_visible_page,rewrite_title_meta
6,content_e5f459e737b7,transactional,page_1,56363,5.9,0.01,-0.23,2.516105,low_ctr_visible_page,improve_snippet_signals
7,content_c1fe78bc4e37,commercial,page_1,134055,7.5,0.03,-0.21,2.479263,low_ctr_visible_page,improve_snippet_signals
8,content_339b357d04c7,informational,page_1,46879,3.7,0.01,-0.23,2.473730,low_ctr_visible_page,rewrite_title_meta
9,content_65114d89496d,transactional,page_1,72631,6.5,0.02,-0.22,2.462495,low_ctr_visible_page,improve_snippet_signals


## 7. Artifacts the paper embeds

The CTR-by-position-tier chart, the core evidence behind the whole lane choice, and the action
mix chart from the Week 7 playbook, both generated fresh here for the deployed paper.


In [7]:
tier_ctr = df[df["impressions_90d"] >= 100].groupby("position_tier")["ctr"].mean()
tier_order = ["page_1", "top_3", "striking", "page_3_5", "deep"]
tier_ctr = tier_ctr.reindex(tier_order)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tier_ctr.index, tier_ctr.values, color="#55A868")
ax.set_ylabel("Mean CTR (%)")
ax.set_xlabel("Position tier")
ax.set_title("CTR by position tier (impressions_90d >= 100)")
plt.tight_layout()
plt.savefig(FIG_DIR / "ctr_by_position_tier.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(6, 4))
action_counts = queue["action"].value_counts()
ax.bar(action_counts.index, action_counts.values, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Pages in queue")
ax.set_title("Action mix across the ranked queue")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "action_mix.png", dpi=150)
plt.close()

print("Artifacts written to:", FIG_DIR)
for f in sorted(FIG_DIR.glob("*.png")):
    print(" -", f.name)
for f in sorted(FIG_DIR.parent.glob("capstone_*.csv")):
    print(" -", f.name)


Artifacts written to: /home/claude/ML-Internship/work/outputs/figures
 - action_mix.png
 - ctr_by_position_tier.png
 - model_vs_baseline_auc.png
 - capstone_model_vs_baseline.csv
 - capstone_top10_recommendations.csv


## 8. Five-minute demo outline (Week 8 showcase, optional)

**0:00 to 0:30, question:** "FlyRank's `low_ctr_visible_page` flag catches pages that rank well
but earn suspiciously few clicks, it says a page qualifies, not which one to open first. With
hundreds of qualifying pages, an editor still needs a starting point."

**0:30 to 1:30, method:** "A transparent baseline ranks pages by how far their CTR falls below
the median of peers at the same position, that's the case-study flag turned into a queue. Then a
Random Forest, deliberately blind to CTR itself, tests whether content and engagement signals
alone can anticipate the same problem, validated on a client-grouped split so it's never tested
on a client it's already seen."

**1:30 to 2:30, one chart:** show Fig. 2 (CTR by position tier), point at the clean drop from
0.35% at page one to 0.055% deep, "this is the pattern the whole baseline is built on, and it's
strong enough to trust at every tier, thousands of pages each."

**2:30 to 4:00, one honest result:** show the model-vs-baseline table. "The baseline hits near
1.000 precision, but that's not a fair comparison, it's built directly from the same quantity
the label thresholds on. The honest number is Random Forest against the dummy floor: 0.727 vs
0.500. Real, but modest, led by engagement rate, not CTR itself." Mention the confession test,
adding `ctr_gap` back in collapses the honest model to that same trivial 1.000, proof the
exclusion mattered.

**4:00 to 5:00, one recommendation:** show the no-go callout, "1,215 pages in this queue have
zero clicks despite huge impression volume, that's not a content problem, that's a tracking bug
waiting to be caught, which is exactly why nothing here auto-rewrites without a human opening
the page first."


## 9. Shareable cuts

**Social post (methodology-focused):**

> Most CTR analyses stop at "low CTR = bad title." Ours started there and then asked a harder
> question: is a global CTR threshold even fair, when a page ranking #1 and a page ranking #40
> get compared against the same number? Turns out no, a naive `ctr < 0.2` rule flags 44% of
> page-one pages and 91% of deep-result pages, the same cutoff means something completely
> different at each position. So we built a position-tier-adjusted score instead, then spent
> just as much effort trying to break our own model: a client-grouped validation split cut our
> apparent precision from a suspicious 1.000 down to an honest 0.700, and a deliberate "confession
> test" caught our own label leakage before it shipped. Full writeup, real numbers, honest
> limitations: https://jasleen13.github.io/ML-Internship/

**Employer-facing summary (3 sentences):**

> I built a position-adjusted CTR opportunity scorer and a validated Random Forest classifier for
> search content prioritization, on FlyRank's real production search dataset (30,000 pages,
> 28 clients, drawn from a 79-million-row warehouse). The honest model reaches ROC AUC 0.727
> against a 0.500 baseline using only content and engagement signals, deliberately excluding
> CTR itself to avoid a label-leakage trap I demonstrated explicitly. The project includes a full
> leakage audit, a client-grouped validation design, and a deployed public research paper with
> reproducible notebooks: https://jasleen13.github.io/ML-Internship/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
